### Analiza pravičnosti modela

In [ ]:
import pandas as pd
import joblib

df = pd.read_csv("../data/processed/final.csv")
model = joblib.load("../models/final_model.pkl")
threshold = joblib.load("../models/threshold.pkl")

In [2]:
df['target']=df['final_result'].isin(["Fail","Withdrawn"]).astype(int)
df['year']=df['code_presentation'].astype(str).str[:4]

In [3]:
train_df=df[df['year']=="2013"].copy()
test_df=df[df['year']=="2014"].copy()

X_train=train_df.drop(columns=['final_result','target','id_student','year'], errors='ignore')
y_train=train_df['target']

X_test=test_df.drop(columns=['final_result','target','id_student','year'], errors='ignore')
y_test=test_df['target']

In [4]:
y_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= threshold).astype(int)

fairness_df = test_df.copy()
fairness_df["y_true"] = y_test.values
fairness_df["y_pred"] = y_pred
fairness_df["risk_probability"] = y_proba

fairness_df.head()

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,submission_rate_TMA,submission_rate_CMA,total_clicks,last_active_day,unique_activities,target,year,y_true,y_pred,risk_probability
358,AAA,2014J,6516,M,Scotland,HE Qualification,80-90%,55<=,0,60,...,1.0,0.0,1296.0,87.0,7.0,0,2014,0,0,0.265252
359,AAA,2014J,24734,F,South Region,Lower Than A Level,60-70%,0-35,0,60,...,1.0,0.0,189.0,86.0,6.0,0,2014,0,1,0.831003
360,AAA,2014J,26192,F,East Anglian Region,A Level or Equivalent,60-70%,35-55,0,60,...,1.0,0.0,1167.0,90.0,7.0,0,2014,0,0,0.050245
361,AAA,2014J,28061,M,Yorkshire Region,Lower Than A Level,40-50%,35-55,0,60,...,1.0,0.0,942.0,88.0,7.0,0,2014,0,0,0.182801
362,AAA,2014J,31600,F,South West Region,A Level or Equivalent,40-50%,0-35,0,60,...,1.0,0.0,272.0,53.0,6.0,0,2014,0,0,0.327259


In [5]:
from sklearn.metrics import accuracy_score, precision_score,recall_score, f1_score

def metrike_po_grupama(df, group_col):
    results = []
    for group_value, group_data in df.groupby(group_col):
        y_true_group = group_data["y_true"]
        y_pred_group = group_data["y_pred"]

        results.append({
            "atribut": group_col,
            "grupa": group_value,
            "broj_studenata": len(group_data),
            "accuracy": accuracy_score(y_true_group, y_pred_group),
            "precision": precision_score(y_true_group, y_pred_group, zero_division=0),
            "recall": recall_score(y_true_group, y_pred_group, zero_division=0),
            "f1": f1_score(y_true_group, y_pred_group, zero_division=0)

        })

    return pd.DataFrame(results)

In [6]:
fairness_gender = metrike_po_grupama(fairness_df, "gender")
fairness_gender

,atribut,grupa,broj_studenata,accuracy,precision,recall,f1
0,gender,F,6252,0.739763,0.637422,0.703212,0.668703
1,gender,M,8255,0.776620,0.728343,0.718797,0.723538


In [7]:
fairness_disability = metrike_po_grupama(fairness_df, "disability")
fairness_disability

,atribut,grupa,broj_studenata,accuracy,precision,recall,f1
0,disability,N,13173,0.760799,0.681504,0.704343,0.692735
1,disability,Y,1334,0.760120,0.742984,0.775039,0.758673


In [8]:
fairness_age = metrike_po_grupama(fairness_df, "age_band")
fairness_age

,atribut,grupa,broj_studenata,accuracy,precision,recall,f1
0,age_band,0-35,10073,0.765115,0.708275,0.730473,0.719202
1,age_band,35-55,4326,0.750578,0.637635,0.663360,0.650243
2,age_band,55<=,108,0.759259,0.578947,0.687500,0.628571


In [9]:
fairness_education = metrike_po_grupama(fairness_df, "highest_education")
fairness_education

,atribut,grupa,broj_studenata,accuracy,precision,recall,f1
0,highest_education,A Level or Equivalent,6456,0.762546,0.672109,0.658452,0.665211
1,highest_education,HE Qualification,2330,0.769528,0.627477,0.650685,0.638870
2,highest_education,Lower Than A Level,5402,0.754535,0.720497,0.778523,0.748387
3,highest_education,No Formal quals,129,0.759690,0.753425,0.808824,0.780142
4,highest_education,Post Graduate Qualification,190,0.768421,0.535714,0.625000,0.576923


In [10]:
fairness_imd = metrike_po_grupama(fairness_df, "imd_band")
fairness_imd

,atribut,grupa,broj_studenata,accuracy,precision,recall,f1
0,imd_band,0-10%,1343,0.731199,0.691995,0.791925,0.738595
1,imd_band,10-20,1449,0.762595,0.731302,0.778761,0.754286
2,imd_band,20-30%,1551,0.756286,0.735849,0.721195,0.728448
3,imd_band,30-40%,1557,0.755941,0.688394,0.701783,0.695024
4,imd_band,40-50%,1413,0.766454,0.693262,0.713504,0.703237
5,imd_band,50-60%,1414,0.764498,0.694158,0.722719,0.708151
6,imd_band,60-70%,1358,0.753314,0.657258,0.663951,0.660588
7,imd_band,70-80%,1341,0.764355,0.695918,0.671260,0.683367
8,imd_band,80-90%,1309,0.760886,0.636364,0.646651,0.641466
9,imd_band,90-100%,1193,0.777871,0.620948,0.687845,0.652687


In [11]:
fairness_region = metrike_po_grupama(fairness_df, "region")
fairness_region

,atribut,grupa,broj_studenata,accuracy,precision,recall,f1
0,region,East Anglian Region,1417,0.736768,0.632509,0.684512,0.657484
1,region,East Midlands Region,1025,0.765854,0.703125,0.681818,0.692308
2,region,Ireland,566,0.766784,0.682927,0.676329,0.679612
3,region,London Region,1346,0.747400,0.676948,0.747312,0.710392
4,region,North Region,828,0.774155,0.683274,0.662069,0.672504
5,region,North Western Region,1211,0.762180,0.706780,0.783835,0.743316
6,region,Scotland,1833,0.756683,0.735294,0.671355,0.701872
7,region,South East Region,930,0.766667,0.651596,0.740181,0.693069
8,region,South Region,1386,0.772727,0.649299,0.698276,0.672897
9,region,South West Region,1074,0.787709,0.723077,0.701493,0.712121


Model okazuje relativno stabilne performanse, ali ne potpuno jednake rezultate za sve posmatrane grupe. 